In [1]:
import sqlite3
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Create (or connect to) a database file
conn = sqlite3.connect('/Users/milenacampos/Documents/projeto doc/DEFEITOS NELORE/nellore_phenotypes.db')
logging.info("Connected to database!")
print(type(conn))

2026-05-05 14:08:20,980 - INFO - Connected to database!


<class 'sqlite3.Connection'>


In [3]:
# Check for duplicate column names
print("Duplicate columns:", df.columns[df.columns.duplicated()].tolist())

# Rename datN to datN_original to avoid conflict
df = df.rename(columns={'datN': 'datN_original'})

# Now write to database
df.to_sql('phenotypes', conn, if_exists='replace', index=False)
logging.info("Table 'phenotypes' written to database!")

Duplicate columns: []


2026-05-05 14:14:47,615 - INFO - Table 'phenotypes' written to database!


In [4]:
# R: dbGetQuery(conn, "SELECT animal, sexo_x, pesod, pesos FROM phenotypes LIMIT 5")
result = pd.read_sql("""
    SELECT animal, "sexo.x", peson, pesod, pesos 
    FROM phenotypes 
    LIMIT 5
""", conn)
print(result)

                animal sexo.x  peson  pesod  pesos
0  GNE23UFBAM000482254      M   26.0  221.0  342.0
1  GNE23UFBAF000223265      F   27.0  203.0  307.0
2  GNE23UFBAF000223278      F   34.0  227.0  322.0
3  GNE23UFBAF000223308      F   29.0  166.0    NaN
4  GNE23UFBAM000482294      M   26.0  206.0  311.0


In [5]:
# R: dbGetQuery(conn, "SELECT * FROM phenotypes WHERE pesod > 300")
heavy = pd.read_sql("""
    SELECT animal, "sexo.x", pesod 
    FROM phenotypes 
    WHERE pesod > 300
    ORDER BY pesod DESC
    LIMIT 10
""", conn)
print(f"Top 10 heaviest at weaning:")
print(heavy)

Top 10 heaviest at weaning:
                animal sexo.x  pesod
0  GNE23UFBAM000474514      M  380.0
1  GNE23UFBAF001003091      F  379.0
2  GNE23UFBAM001069077      M  373.0
3  GNE23UFBAM000474066      M  371.0
4  GNE23UFBAM000867469      M  370.0
5  GNE23UFBAM000474372      M  369.0
6  GNE23UFBAM001069468      M  368.0
7  GNE23UFBAM001068854      M  366.0
8  GNE23UFBAM000948649      M  365.0
9  GNE23UFBAM001160764      M  364.0


In [6]:
# R: equivalent of group_by(sexo) %>% summarise(n=n(), mean=mean(pesod))
summary = pd.read_sql("""
    SELECT "sexo.x",
           COUNT(*) as n_animals,
           ROUND(AVG(pesod), 1) as mean_weaning,
           ROUND(AVG(pesos), 1) as mean_yearling
    FROM phenotypes
    WHERE pesod IS NOT NULL
    GROUP BY "sexo.x"
""", conn)
print(summary)

  sexo.x  n_animals  mean_weaning  mean_yearling
0      F     177774         175.6          267.3
1      M     182383         190.4          315.0


In [8]:
# Create a small sire summary table
sire_summary = pd.read_sql("""
    SELECT touro as sire,
           COUNT(*) as n_offspring,
           ROUND(AVG(pesod), 1) as mean_weaning_offspring
    FROM phenotypes
    WHERE pesod IS NOT NULL
    GROUP BY touro
    HAVING COUNT(*) >= 100
    ORDER BY n_offspring DESC
    LIMIT 20
""", conn)

# Write it as a second table in the database
sire_summary.to_sql('sire_summary', conn, if_exists='replace', index=False)
print(f"Created sire_summary table with {len(sire_summary)} sires")
print(sire_summary.head())

Created sire_summary table with 20 sires
                  sire  n_offspring  mean_weaning_offspring
0  GNE23UFBAM000560100        17850                   190.7
1  GNE23UFBAM000534108        16452                   178.5
2  GNE23UFBAM000895809         7631                   187.4
3  GNE23UFBAM001104061         6411                   187.1
4  GNE23UFBAM000211857         4636                   192.1


In [9]:
# Join phenotypes with sire summary
# R equivalent: left_join(phenotypes, sire_summary, by=c("touro"="sire"))
joined = pd.read_sql("""
    SELECT p.animal, p."sexo.x", p.pesod, 
           s.n_offspring, s.mean_weaning_offspring
    FROM phenotypes p
    LEFT JOIN sire_summary s ON p.touro = s.sire
    WHERE p.pesod IS NOT NULL
    LIMIT 10
""", conn)
print(joined)

                animal sexo.x  pesod n_offspring mean_weaning_offspring
0  GNE23UFBAM000482254      M  221.0        None                   None
1  GNE23UFBAF000223265      F  203.0        None                   None
2  GNE23UFBAF000223278      F  227.0        None                   None
3  GNE23UFBAF000223308      F  166.0        None                   None
4  GNE23UFBAM000482294      M  206.0        None                   None
5  GNE23UFBAF000223338      F  226.0        None                   None
6  GNE23UFBAF000223352      F  204.0        None                   None
7  GNE23UFBAF000223360      F  200.0        None                   None
8  GNE23UFBAF000223384      F  190.0        None                   None
9  GNE23UFBAM000482349      M  180.0        None                   None


In [10]:
# Find animals whose sire IS in our top 20
top_sire = 'GNE23UFBAM000560100'  # most prolific sire from our table

result = pd.read_sql(f"""
    SELECT p.animal, p."sexo.x", p.pesod,
           s.n_offspring, s.mean_weaning_offspring
    FROM phenotypes p
    LEFT JOIN sire_summary s ON p.touro = s.sire
    WHERE p.touro = '{top_sire}'
    AND p.pesod IS NOT NULL
    LIMIT 5
""", conn)
print(result)

                animal sexo.x  pesod  n_offspring  mean_weaning_offspring
0  GNE23UFBAM000491015      M  154.0        17850                   190.7
1  GNE23UFBAM000473952      M  222.0        17850                   190.7
2  GNE23UFBAM000474087      M  222.0        17850                   190.7
3  GNE23UFBAM000474400      M  186.0        17850                   190.7
4  GNE23UFBAM000474482      M  210.0        17850                   190.7


In [11]:
# Final query — farm performance summary
# Combines SELECT + WHERE + GROUP BY + HAVING + ORDER BY
farm_summary = pd.read_sql("""
    SELECT fazd as farm,
           "sexo.x" as sex,
           COUNT(*) as n_animals,
           ROUND(AVG(pesod), 1) as mean_weaning,
           ROUND(AVG(pesos), 1) as mean_yearling,
           ROUND(MIN(pesod), 1) as min_weaning,
           ROUND(MAX(pesod), 1) as max_weaning
    FROM phenotypes
    WHERE pesod IS NOT NULL
    AND "sexo.x" IN ('M', 'F')
    GROUP BY fazd, "sexo.x"
    HAVING COUNT(*) >= 500
    ORDER BY mean_weaning DESC
    LIMIT 10
""", conn)
print(farm_summary)

# Close connection when done
conn.close()
logging.info("Database connection closed.")

2026-05-05 15:55:43,943 - INFO - Database connection closed.


      farm sex  n_animals  mean_weaning  mean_yearling  min_weaning  \
0  faz0042   M        705         232.5          523.7         90.0   
1  faz0090   M        769         229.1          339.8         75.0   
2  faz0122   M        720         222.3          358.1         90.0   
3  faz0055   M        802         221.3          326.0        110.0   
4  faz0071   M        518         220.5          273.5        105.0   
5  faz0101   M        670         220.0          319.8        104.0   
6  faz0060   M       2046         217.9          356.7         81.0   
7  faz0047   M       1334         216.0          331.1        100.0   
8  faz0035   M       1105         215.7          378.3        107.0   
9  faz0056   M       1727         215.6          338.2        100.0   

   max_weaning  
0        331.0  
1        314.0  
2        350.0  
3        304.0  
4        324.0  
5        328.0  
6        328.0  
7        331.0  
8        314.0  
9        316.0  
